In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col, regexp_extract

landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/data/"
schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/fred_raw_json/"

@dlt.table(
    name="bronze_fred_raw_json",
    comment="Raw FRED API payloads as JSON (one row per file)"
)
def bronze_fred_raw_json():
    return (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .option("multiLine", "true")
          .load(landing_path)
          .withColumn("_ingest_ts", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
          .withColumn("series_id", regexp_extract(col("_metadata.file_path"), r"/data/([^/]+)/", 1))
    )